# Multi-hole spectral clustering

## What this notebook does

This notebook builds **one shared spectral clustering model** across several HyLogger drill holes. All eligible NIR/SWIR spectra are combined before PCA and K-Means, so a cluster number has the same meaning in every hole. The original hole ID, depth, and TSG scalar classifications remain attached to every result row.

The clusters are created from spectral shape only. `Min1_sTSAS`, `Grp1_sTSAS`, `Min1_uTSAS`, and `Grp1_uTSAS` are not model inputs; they are used afterward to interpret the independently created clusters.

### Workflow

1. Choose drill holes and required scalar columns.
2. Check or download each raw HyLogger package.
3. Exclude holes that do not contain all required scalar columns.
4. Align wavelengths, apply SNV, reduce dimensions with PCA, test candidate K values, and fit one shared K-Means model.
5. Compare cluster proportions and mean spectral shapes.
6. Interpret clusters with `Grp1_sTSAS`.
7. Compare whole-hole mineral profiles and exploratory depth intervals.
8. Save reusable outputs and timestamped Markdown reports.

Raw packages must be available through `hylogger raw fetch <HOLE_ID> --components spectra`.

In [5]:
# Import the standard libraries and reusable HyLogger pipeline functions.
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from hylogger.clustering import cluster_holes, load_holes_with_scalar_check
from hylogger.config import AppConfig

## 1. Choose the drill-hole cohort and model settings

`HOLE_IDS` controls which drill holes enter this run. Add or remove an uppercase hole ID here; do not include `.zip`.

`REQUIRED_SCALAR_COLUMNS` is a data-quality gate. A hole proceeds only when all four columns exist. These labels are retained for interpretation and are not used to form the spectral clusters.

`K_VALUES` defines the candidate numbers of clusters. The pipeline tests every value instead of hardcoding K. `TRAINING_SAMPLE_SIZE=None` means PCA and final K-Means use every finite spectrum. `RANDOM_STATE` makes repeated runs reproducible.

In [6]:
# Define the cohort, required interpretation columns, K search range, and reproducibility settings.
HOLE_IDS = [
    "05GJD001",
    "05KCD001",
    "CDD001",
]
REQUIRED_SCALAR_COLUMNS = [
    "Min1_sTSAS",
    "Grp1_sTSAS",
    "Min1_uTSAS",
    "Grp1_uTSAS",
]
K_VALUES = range(2, 9)
TRAINING_SAMPLE_SIZE = None  # None fits PCA and K-Means using every spectrum.
RANDOM_STATE = 42

## 2. Check or download raw packages from NCI THREDDS

This block calls the project's existing `hylogger raw fetch` command once per selected hole.

- `DOWNLOAD_RAW_PACKAGES=False` performs a dry run: it shows the URL, complete ZIP size, and target path without downloading.
- `DOWNLOAD_RAW_PACKAGES=True` downloads or resumes the complete ZIP and extracts only spectral components.
- Completed packages are registered under `data/downloads/raw/<HOLE_ID>/`. An interrupted download resumes when the cell runs again.

The service provides one complete ZIP per hole, so `--components spectra` reduces extraction but not download size. Already completed packages are reused.

In [ ]:
# Use the project CLI to dry-run or download every selected raw package.

DOWNLOAD_RAW_PACKAGES = True  # False = show sizes only; True = download

for hole_id in HOLE_IDS:
    command = [
        sys.executable, "-m", "hylogger", "raw", "fetch",
        hole_id, "--components", "spectra",
    ]
    if not DOWNLOAD_RAW_PACKAGES:
        command.append("--dry-run")
    print(f"\n{'Downloading' if DOWNLOAD_RAW_PACKAGES else 'Checking'} {hole_id}...")
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    if completed.returncode != 0:
        print(f"Could not fetch {hole_id}; continuing with the other holes.")


Checking PND001...
[1/1] Raw package PND001
  URL: https://thredds.nci.org.au/thredds/fileServer/rs07/WA/PND001.zip
  Size: 1.6 GiB
  Target: C:\Users\sathi\OneDrive\Desktop\group_hyloger\gswa-hylogger-data-crunch\data\downloads\raw\PND001\archive\PND001.zip
  Components: spectra

Checking 05GJD001...
[1/1] Raw package 05GJD001
  URL: https://thredds.nci.org.au/thredds/fileServer/rs07/WA/05GJD001.zip
  Size: 746.2 MiB
  Target: C:\Users\sathi\OneDrive\Desktop\group_hyloger\gswa-hylogger-data-crunch\data\downloads\raw\05GJD001\archive\05GJD001.zip
  Components: spectra

Checking 05KCD001...
[1/1] Raw package 05KCD001
  URL: https://thredds.nci.org.au/thredds/fileServer/rs07/WA/05KCD001.zip
  Size: 689.8 MiB
  Target: C:\Users\sathi\OneDrive\Desktop\group_hyloger\gswa-hylogger-data-crunch\data\downloads\raw\05KCD001\archive\05KCD001.zip
  Components: spectra

Checking CDD001...
[1/1] Raw package CDD001
  URL: https://thredds.nci.org.au/thredds/fileServer/rs07/WA/CDD001.zip
  Size: 38.3 

## 3. Load holes and verify required scalar columns

This block opens each extracted package, loads every finite NIR/SWIR spectrum, and checks whether the four requested scalar columns exist.

- `scalar_check` is a readable table showing package errors, column availability, and eligibility.
- `eligible_holes` contains the loaded holes that passed every check and will enter clustering.

Mask values do not filter rows; both mask states are included. At least two eligible holes are required because this is a cross-hole model.

In [ ]:
# Load packages, verify scalar availability, and keep only holes ready for shared clustering.
eligible_holes, scalar_check = load_holes_with_scalar_check(
    HOLE_IDS,
    required_columns=REQUIRED_SCALAR_COLUMNS,
)
display(scalar_check)
print("Eligible holes:", [hole.hole_id for hole in eligible_holes])
if len(eligible_holes) < 2:
    raise ValueError("At least two downloaded holes must contain all four scalar columns.")

## 4. Fit the shared spectral pipeline and select K

This is the main modelling block:

1. **Combine holes:** stack all eligible spectra while preserving hole ID and depth.
2. **Align wavelengths:** resample holes to their common NIR/SWIR wavelength grid.
3. **SNV normalisation:** normalise each spectrum independently so clustering focuses more on spectral shape than brightness.
4. **PCA:** reduce the 531 wavelength columns while retaining 95% of spectral variance. PCA and final K-Means use all spectra when `TRAINING_SAMPLE_SIZE=None`.
5. **Balanced diagnostics:** take a capped, approximately equal sample from each hole so large holes do not dominate K selection. This sample is used only for diagnostics; every spectrum receives a final cluster.
6. **Test K:** fit K-Means for every value in `K_VALUES`.
7. **Consensus selection:** rank each K using silhouette, Calinski–Harabasz, Davies–Bouldin, inertia elbow strength, repeated-seed stability, minimum cluster size, and cross-hole coverage. Solutions with a cluster below 1% or insufficient hole coverage fail the quality gate. The valid K with the highest consensus score is selected.

`RANDOM_STATE=42` does not choose K randomly. It makes sampling and K-Means initialisation repeatable. Scalar mineral labels remain attached for interpretation but do not influence clustering.

In [4]:
# Run preprocessing, balanced K evaluation, consensus selection, and final shared clustering.
result = cluster_holes(
    eligible_holes,
    n_clusters=None,  # Select K from the test scores below.
    k_values=K_VALUES,
    variance_retained=0.95,
    training_sample_size=TRAINING_SAMPLE_SIZE,
    silhouette_sample_size=5_000,  # Balanced equally across holes.
    stability_repeats=3,
    minimum_cluster_percentage=1.0,
    minimum_hole_coverage=0.5,
    random_state=RANDOM_STATE,
)
print(f"PCA components: {result.pca.n_components_}")
print(f"Clustered spectra: {len(result.assignments):,}")
print(f"Consensus-selected K: {result.model.n_clusters}")
display(result.k_scores)
display(result.cluster_sizes)

NameError: name 'cluster_holes' is not defined

## 5. Compare cluster proportions between holes

This stacked bar chart shows what percentage of each hole's spectra belongs to each shared cluster. Because one model was fitted across all holes, Cluster 0 is directly comparable between holes, as are the other cluster numbers. The chart reveals holes dominated by similar or different spectral patterns; it does not assign geological names to clusters.

In [ ]:
# Reshape per-hole cluster percentages and draw a comparable stacked bar chart.
composition = result.cluster_sizes.pivot(
    index="hole_id", columns="cluster_id", values="percentage"
).fillna(0)
composition.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="tab10")
plt.ylabel("Spectra (%)")
plt.title("Shared spectral clusters by hole")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Inspect the mean spectral shape of each cluster

For each selected cluster, this block averages the SNV-normalised spectra assigned to it and plots the mean against wavelength. Differences between curves show which spectral-shape features separate clusters. Cluster numbers are arbitrary mathematical labels and should be interpreted with spectral features, depth continuity, and independent geological information.

In [ ]:
# Plot the average normalised spectrum that characterises each shared cluster.
plt.figure(figsize=(12, 6))
for cluster_id, spectrum in enumerate(result.mean_snv_spectra):
    plt.plot(result.wavelengths, spectrum, label=f"Cluster {cluster_id}")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Mean SNV-normalised reflectance")
plt.title("Mean spectral shape for shared clusters")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7. Interpret each hole–cluster combination with Grp1 sTSAS

This block uses only the system-generated `Grp1_sTSAS` classification. Within each drill hole and spectral cluster, it counts every mineral-group label and converts the counts to percentages. It answers questions such as: “Within Cluster 2 of 05GJD001, what proportion is classified as chlorite or white mica?”

`BLANK`, `INVALID`, and other labels remain visible so classification coverage is not hidden. Separate Markdown blocks prevent VS Code from truncating later clusters. Every run also creates a timestamped report and refreshes a `latest` copy under `data_view/outputs/mineral_reports/`.

These percentages help interpret clusters; they do not prove that a cluster represents one mineral group.

In [ ]:
# Calculate readable Grp1_sTSAS percentages for every hole–cluster pair and save a report.
from datetime import datetime

from IPython.display import Markdown, display

labels = (
    result.assignments["Grp1_sTSAS"]
    .astype("string")
    .str.strip()
    .fillna("BLANK")
    .replace("", "BLANK")
)

group_counts = pd.crosstab(
    [result.assignments["hole_id"], result.assignments["cluster_id"]],
    labels,
)
clustered_hole_ids = result.assignments["hole_id"].drop_duplicates().tolist()
all_hole_clusters = pd.MultiIndex.from_product(
    [clustered_hole_ids, range(result.model.n_clusters)],
    names=["hole_id", "cluster_id"],
)
group_counts = group_counts.reindex(all_hole_clusters, fill_value=0)
group_percentages = group_counts.div(group_counts.sum(axis=1), axis=0).mul(100)

report_lines = [
    "# Grp1 sTSAS percentages by drill hole and cluster",
    "",
]

for hole_id in clustered_hole_ids:
    hole_heading = f"## Drill hole `{hole_id}`"
    report_lines.extend([hole_heading, ""])
    display(Markdown(hole_heading))
    for cluster_id in range(result.model.n_clusters):
        percentages = group_percentages.loc[(hole_id, cluster_id)]
        lines = [f"### Cluster {cluster_id}"]
        for mineral_group, percentage in percentages.sort_values(ascending=False).items():
            if percentage > 0:
                lines.append(f"- **{mineral_group}**: {percentage:.2f}%")
        if len(lines) == 1:
            lines.append("- No spectra assigned to this cluster")
        report_lines.extend(lines + [""])
        display(Markdown("\n".join(lines)))

output_folder = (
    AppConfig.load().repo_root / "data_view" / "outputs" / "mineral_reports"
)
output_folder.mkdir(parents=True, exist_ok=True)
report_text = "\n".join(report_lines)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mineral_report_path = (
    output_folder / f"grp1_stsas_by_hole_and_cluster_{run_timestamp}.md"
)
latest_report_path = output_folder / "grp1_stsas_by_hole_and_cluster_latest.md"
mineral_report_path.write_text(report_text, encoding="utf-8")
latest_report_path.write_text(report_text, encoding="utf-8")
print(f"Saved this run: {mineral_report_path.resolve()}")
print(f"Updated latest report: {latest_report_path.resolve()}")

## 8. Compare and group whole drill holes using Grp1 sTSAS

This block collapses each complete hole into one overall `Grp1_sTSAS` percentage profile. Cosine similarity compares two profiles: 100% means identical relative composition, while lower values indicate greater difference. For every hole pair, the output lists strongly shared groups and the largest percentage-point differences.

Hierarchical clustering groups holes with similar overall profiles. Candidate group counts are evaluated with silhouette score when enough holes exist. This is exploratory—not proof that grouped holes are the same geological unit. Large `BLANK` proportions and automated sTSAS classifications can influence results, so geological and spatial validation is still required.

In [ ]:
# Compare whole-hole mineral-group profiles and group compositionally similar holes.
from itertools import combinations

from IPython.display import Markdown, display
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

hole_group_counts = pd.crosstab(result.assignments["hole_id"], labels)
hole_group_percentages = (
    hole_group_counts.div(hole_group_counts.sum(axis=1), axis=0).mul(100)
)

similarity = pd.DataFrame(
    cosine_similarity(hole_group_percentages),
    index=hole_group_percentages.index,
    columns=hole_group_percentages.index,
).mul(100)
display(Markdown("### Pairwise sTSAS mineral-group similarity (%)"))
display(similarity.round(1))

display(Markdown("### Where each pair is most similar and different"))
for first_hole, second_hole in combinations(hole_group_percentages.index, 2):
    first = hole_group_percentages.loc[first_hole]
    second = hole_group_percentages.loc[second_hole]
    absolute_difference = (first - second).abs()
    shared = pd.concat([first, second], axis=1).min(axis=1)
    shared = shared[shared > 0].sort_values(ascending=False).head(5)
    differences = absolute_difference.sort_values(ascending=False).head(5)
    lines = [
        f"#### `{first_hole}` compared with `{second_hole}`",
        f"**Overall similarity: {similarity.loc[first_hole, second_hole]:.1f}%**",
        "",
        "**Strongest shared mineral groups**",
    ]
    for mineral_group, _shared_percentage in shared.items():
        lines.append(
            f"- **{mineral_group}**: {first_hole} {first[mineral_group]:.1f}% | "
            f"{second_hole} {second[mineral_group]:.1f}%"
        )
    lines.extend(["", "**Largest differences**"])
    for mineral_group, difference in differences.items():
        lines.append(
            f"- **{mineral_group}**: {first_hole} {first[mineral_group]:.1f}% | "
            f"{second_hole} {second[mineral_group]:.1f}% "
            f"(difference {difference:.1f} percentage points)"
        )
    display(Markdown("\n".join(lines)))

number_of_holes = len(hole_group_percentages)
if number_of_holes >= 3:
    grouping_scores = []
    grouping_candidates = {}
    for number_of_groups in range(2, min(6, number_of_holes - 1) + 1):
        grouping_model = AgglomerativeClustering(n_clusters=number_of_groups)
        grouping_labels = grouping_model.fit_predict(hole_group_percentages)
        grouping_score = silhouette_score(
            hole_group_percentages, grouping_labels, metric="cosine"
        )
        grouping_scores.append(
            {"number_of_groups": number_of_groups, "silhouette_score": grouping_score}
        )
        grouping_candidates[number_of_groups] = grouping_labels
    grouping_scores = pd.DataFrame(grouping_scores)
    selected_groups = int(
        grouping_scores.loc[grouping_scores["silhouette_score"].idxmax(), "number_of_groups"]
    )
    hole_groups = pd.DataFrame(
        {
            "hole_id": hole_group_percentages.index,
            "similar_hole_group": grouping_candidates[selected_groups],
        }
    ).sort_values(["similar_hole_group", "hole_id"])
    display(Markdown(f"### Similar drill-hole groups ({selected_groups} groups selected)"))
    for group_id, group in hole_groups.groupby("similar_hole_group"):
        holes = ", ".join(f"`{hole}`" for hole in group["hole_id"])
        display(Markdown(f"- **Group {group_id}**: {holes}"))
    display(Markdown("#### Drill-hole grouping diagnostics"))
    display(grouping_scores.round(3))
else:
    display(Markdown("At least three holes are required for automatic grouping."))

## 9. Find exploratory Grp1 sTSAS similarities between depth intervals

This block divides every hole into configurable 5 m bins. Within each bin it calculates the `Grp1_sTSAS` percentage profile, then compares bins from different holes using cosine similarity.

- `DEPTH_INTERVAL_METRES` controls bin width.
- `MINIMUM_READINGS_PER_INTERVAL` prevents very small bins from being compared.
- `TOP_MATCHES_PER_HOLE_PAIR` limits report length.
- `EXCLUDED_GROUP_LABELS` removes `BLANK`, `INVALID`, and `NOTAROK` so missing classifications do not create false similarity.

A high score means only that two intervals have similar sTSAS mineral-group proportions. It does not establish stratigraphic equivalence: this method does not use spatial position, collar elevation, lithology, spectral-cluster proportions, sequence continuity, or one-to-one interval matching. Treat it as exploratory. Each run is saved under `data_view/outputs/depth_similarity_reports/`.

In [ ]:
# Compare Grp1_sTSAS profiles in 5 m bins across holes and save the top interval matches.
from datetime import datetime
from itertools import combinations

import numpy as np
from IPython.display import Markdown, display
from sklearn.metrics.pairwise import cosine_similarity

DEPTH_INTERVAL_METRES = 5
MINIMUM_READINGS_PER_INTERVAL = 10
TOP_MATCHES_PER_HOLE_PAIR = 10
EXCLUDED_GROUP_LABELS = {"BLANK", "INVALID", "NOTAROK"}

depth_data = result.assignments.copy()
depth_data["depth_m"] = pd.to_numeric(depth_data["depth_m"], errors="coerce")
depth_data = depth_data.dropna(subset=["depth_m"])
depth_data["depth_start_m"] = (
    np.floor(depth_data["depth_m"] / DEPTH_INTERVAL_METRES)
    * DEPTH_INTERVAL_METRES
)
depth_data["Grp1_sTSAS_clean"] = (
    depth_data["Grp1_sTSAS"]
    .astype("string")
    .str.strip()
    .fillna("BLANK")
    .replace("", "BLANK")
)
depth_data = depth_data[
    ~depth_data["Grp1_sTSAS_clean"].isin(EXCLUDED_GROUP_LABELS)
]

interval_counts = pd.crosstab(
    [depth_data["hole_id"], depth_data["depth_start_m"]],
    depth_data["Grp1_sTSAS_clean"],
)
interval_totals = interval_counts.sum(axis=1)
interval_counts = interval_counts.loc[
    interval_totals >= MINIMUM_READINGS_PER_INTERVAL
]
interval_profiles = interval_counts.div(interval_counts.sum(axis=1), axis=0)

depth_report_lines = [
    "# Most similar Grp1 sTSAS depth intervals",
    "",
    f"Depth interval: {DEPTH_INTERVAL_METRES} m",
    f"Minimum readings per interval: {MINIMUM_READINGS_PER_INTERVAL}",
    "",
]

available_holes = interval_profiles.index.get_level_values("hole_id").unique()
for first_hole, second_hole in combinations(available_holes, 2):
    first_profiles = interval_profiles.xs(first_hole, level="hole_id")
    second_profiles = interval_profiles.xs(second_hole, level="hole_id")
    scores = cosine_similarity(first_profiles, second_profiles) * 100
    candidates = []
    for first_index, first_depth in enumerate(first_profiles.index):
        for second_index, second_depth in enumerate(second_profiles.index):
            candidates.append(
                (scores[first_index, second_index], first_depth, second_depth)
            )
    candidates.sort(reverse=True)
    heading = f"## `{first_hole}` compared with `{second_hole}`"
    lines = [heading]
    for score, first_depth, second_depth in candidates[:TOP_MATCHES_PER_HOLE_PAIR]:
        lines.append(
            f"- **{score:.1f}% similar**: "
            f"{first_hole} {first_depth:g}–{first_depth + DEPTH_INTERVAL_METRES:g} m "
            f"↔ {second_hole} {second_depth:g}–{second_depth + DEPTH_INTERVAL_METRES:g} m"
        )
    display(Markdown("\n".join(lines)))
    depth_report_lines.extend(lines + [""])

depth_report_folder = (
    AppConfig.load().repo_root / "data_view" / "outputs" / "depth_similarity_reports"
)
depth_report_folder.mkdir(parents=True, exist_ok=True)
depth_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
depth_report_text = "\n".join(depth_report_lines)
depth_report_path = (
    depth_report_folder / f"depth_interval_similarity_{depth_timestamp}.md"
)
depth_latest_path = depth_report_folder / "depth_interval_similarity_latest.md"
depth_report_path.write_text(depth_report_text, encoding="utf-8")
depth_latest_path.write_text(depth_report_text, encoding="utf-8")
print(f"Saved depth-interval report: {depth_report_path.resolve()}")

## 10. Save reusable machine-readable outputs

This block saves detailed spectrum assignments as Parquet and the small per-hole cluster-size summary as CSV. Parquet preserves data types and efficiently stores the large row-level result; CSV keeps the summary easy to inspect. These outputs support later analysis without rerunning package parsing and clustering. Formatted mineral and depth-similarity reports are saved separately by sections 7 and 9.

In [ ]:
# Save row-level assignments and the compact cluster-size summary for reuse.
config = AppConfig.load()
output_dir = Path("experiments/multi_hole_initial")
output_dir.mkdir(parents=True, exist_ok=True)
result.assignments.to_parquet(output_dir / "spectrum_assignments.parquet", index=False)
result.cluster_sizes.to_csv(output_dir / "cluster_sizes_by_hole.csv", index=False)
print(f"Saved outputs to {output_dir.resolve()}")